# Notebook 4 — Forecasting: Model Training

Trains all 3 Task 1 models (XGBoost, LightGBM, LSTM) on leakage-safe features. Does NOT evaluate them — that's Notebook 5's job. Every step below has been tested against the real dataset and verified before being included here.

## Step 0 — Imports & Reproducibility

In [1]:
import os, sys, random, time
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import r2_score, mean_absolute_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

SEED = 42
random.seed(SEED); os.environ['PYTHONHASHSEED'] = str(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("pandas:", pd.__version__, "| tensorflow:", tf.__version__)

pandas: 2.2.2 | tensorflow: 2.16.1


## Step 0b — Fix Working Directory

In [2]:
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working directory:", os.getcwd())
os.makedirs("models", exist_ok=True)

Working directory: c:\Users\Dhruvajyoti Bordoloi\Desktop\smartdc-energy\ml-pipeline


## Step 1 — Load

In [3]:
INPUT_PATH = "data/processed/master_features.csv"
assert os.path.exists(INPUT_PATH), f"Missing input: {INPUT_PATH} — run 02_feature_engineering.ipynb first"

df = pd.read_csv(INPUT_PATH, parse_dates=['timestamp']).set_index('timestamp')
print(df.shape)

(26280, 21)


## Step 2 — Lag & Rolling Features

All built on `total_power_kwh` with `.shift(1)` protection, same leakage-safe pattern established in Notebook 2. `lag_168h` (1 week) costs the first 168 rows — verified: 26,280 → 26,112 rows.

In [4]:
TARGET = 'total_power_kwh'
for lag in [1, 6, 12, 24, 48, 168]:
    df[f'lag_{lag}h'] = df[TARGET].shift(lag)
for window in [6, 24]:
    df[f'roll_mean_{window}h'] = df[TARGET].shift(1).rolling(window).mean()
    df[f'roll_std_{window}h'] = df[TARGET].shift(1).rolling(window).std()

before = len(df)
df = df.dropna()
print(f"Rows: {before} -> {len(df)} (lost {before-len(df)}, expected 168)")
assert before - len(df) == 168, "Unexpected row loss — investigate before continuing"

Rows: 26280 -> 26112 (lost 168, expected 168)


## Step 3 — Leakage-Safe Feature Selection & Chronological Split

**`total_power_kwh = it_load_kwh + cooling_load_kwh + support_load_kwh` is an exact arithmetic identity.** Including those columns (or `pue`) as features lets the model 're-predict' via addition instead of learning — verified: excluding them still yields 25 genuine features, and a chronological split confirms zero overlap between train and test periods.

In [5]:
LEAK_COLS = ['it_load_kwh', 'cooling_load_kwh', 'support_load_kwh', 'pue']
feature_cols = [c for c in df.columns if c not in [TARGET, 'anomaly_flag'] + LEAK_COLS]
print(f"Feature count: {len(feature_cols)}")
assert not any(c in feature_cols for c in LEAK_COLS), "LEAKAGE: a forbidden column is still present"

X, y = df[feature_cols], df[TARGET]
split_idx = int(len(df) * 0.85)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Train: {len(X_train)} rows ({X_train.index.min()} to {X_train.index.max()})")
print(f"Test:  {len(X_test)} rows ({X_test.index.min()} to {X_test.index.max()})")
assert X_train.index.max() < X_test.index.min(), "Train/test overlap detected"

Feature count: 25
Train: 22195 rows (2022-01-08 00:00:00 to 2024-07-20 18:00:00)
Test:  3917 rows (2024-07-20 19:00:00 to 2024-12-30 23:00:00)


## Step 4 — Scale (fit on train only)

In [6]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)
print("Scaled shapes:", X_train_sc.shape, X_test_sc.shape)

Scaled shapes: (22195, 25) (3917, 25)


## Step 5 — Train Model 1: XGBoost

**About the printed output:** the last line auto-displays the fitted model's parameter dump — harmless, not an error. Add a semicolon after `.fit(...)` to suppress it if preferred.

In [7]:
xgb_model = XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=6,
                          subsample=0.8, colsample_bytree=0.8, random_state=SEED, n_jobs=-1)
t0 = time.perf_counter()
xgb_model.fit(X_train_sc, y_train)
train_time_xgb = time.perf_counter() - t0
print(f"Trained in {train_time_xgb:.2f}s")

pred_xgb = xgb_model.predict(X_test_sc)
r2_xgb = r2_score(y_test, pred_xgb)
print(f"XGBoost R2: {r2_xgb:.4f}  (realistic range: 0.88-0.95)")
if r2_xgb > 0.98:
    print("WARNING: R2 > 0.98 — check LEAK_COLS above")

Trained in 2.06s
XGBoost R2: 0.8945  (realistic range: 0.88-0.95)


## Step 6 — Train Model 2: LightGBM

In [8]:
lgbm_model = LGBMRegressor(n_estimators=400, learning_rate=0.05, max_depth=6,
                            num_leaves=63, random_state=SEED, verbose=-1)
t0 = time.perf_counter()
lgbm_model.fit(X_train_sc, y_train)
train_time_lgbm = time.perf_counter() - t0
print(f"Trained in {train_time_lgbm:.2f}s")

pred_lgbm = lgbm_model.predict(X_test_sc)
r2_lgbm = r2_score(y_test, pred_lgbm)
print(f"LightGBM R2: {r2_lgbm:.4f}  (realistic range: 0.87-0.95)")
print(f"Difference from XGBoost: {abs(r2_lgbm - r2_xgb):.4f} (should be small — a large gap signals a problem in one of them)")

c:\Users\Dhruvajyoti Bordoloi\Desktop\smartdc-energy\venv\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\Dhruvajyoti Bordoloi\Desktop\smartdc-energy\venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\Dhruvajyoti Bordoloi\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Dhruvajyoti Bordoloi\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, e

Trained in 0.81s
LightGBM R2: 0.8935  (realistic range: 0.87-0.95)
Difference from XGBoost: 0.0010 (should be small — a large gap signals a problem in one of them)


## Step 7 — Train Model 3: LSTM

Separate `MinMaxScaler`s for X and y (standard for LSTM). The windowing offsets `y_test_lstm` by `WINDOW` rows relative to `y_test` — the explicit assertion below catches any misalignment before it can silently corrupt every downstream metric.

In [9]:
mm_scaler = MinMaxScaler()
X_train_mm = mm_scaler.fit_transform(X_train)
X_test_mm = mm_scaler.transform(X_test)
y_scaler = MinMaxScaler()
y_train_mm = y_scaler.fit_transform(y_train.values.reshape(-1,1))

def make_sequences(X, y, window=24):
    Xs, ys = [], []
    for i in range(window, len(X)):
        Xs.append(X[i-window:i]); ys.append(y[i])
    return np.array(Xs), np.array(ys)

WINDOW = 24
X_train_seq, y_train_seq = make_sequences(X_train_mm, y_train_mm, WINDOW)
X_test_seq, _ = make_sequences(X_test_mm, np.zeros((len(X_test_mm),1)), WINDOW)
y_test_lstm = y_test.iloc[WINDOW:]

assert len(X_test_seq) == len(y_test_lstm), "LSTM window alignment mismatch — check indices"
print(f"Sequences built. Train: {X_train_seq.shape}, Test: {X_test_seq.shape}. Alignment OK.")

Sequences built. Train: (22171, 24, 25), Test: (3893, 24, 25). Alignment OK.


In [10]:
lstm_model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(WINDOW, X_train.shape[1])),
    Dropout(0.2), LSTM(32), Dropout(0.2), Dense(1)
])
lstm_model.compile(optimizer='adam', loss='mse')
es = EarlyStopping(patience=8, restore_best_weights=True)

t0 = time.perf_counter()
history = lstm_model.fit(X_train_seq, y_train_seq, epochs=50, batch_size=64,
                          validation_split=0.1, callbacks=[es], verbose=1)
train_time_lstm = time.perf_counter() - t0
print(f"Trained in {train_time_lstm:.2f}s")

c:\Users\Dhruvajyoti Bordoloi\Desktop\smartdc-energy\venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/50
312/312 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - loss: 0.0102 - val_loss: 0.0053
Epoch 2/50
312/312 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 0.0061 - val_loss: 0.0052
Epoch 3/50
312/312 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - loss: 0.0055 - val_loss: 0.0045
Epoch 4/50
312/312 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - loss: 0.0050 - val_loss: 0.0049
Epoch 5/50
312/312 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - loss: 0.0046 - val_loss: 0.0044
Epoch 6/50
312/312 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - loss: 0.0044 - val_loss: 0.0045
Epoch 7/50
312/312 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 0.0042 - val_loss: 0.0044
Epoch 8/50
312/312 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - loss: 0.0041 - val_loss: 0.0052
Epoch 9/50
312/312 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - loss: 0.0040 - val_loss: 0.0050
Epoch 10/50
312/312 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 0.0039 - val_loss: 0.0042
Epoch 11/50
312/312 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 0.0037 - val_loss: 0.0043
Epoch 12/50
312/312 ━━━━━━━━━━━━━━━━━━━━ 

In [11]:
pred_lstm_scaled = lstm_model.predict(X_test_seq)
pred_lstm = y_scaler.inverse_transform(pred_lstm_scaled).flatten()
r2_lstm = r2_score(y_test_lstm, pred_lstm)
print(f"LSTM R2: {r2_lstm:.4f}  (expect roughly 0.82-0.93 — LSTM often trails tree models slightly on this feature set)")

122/122 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
LSTM R2: 0.8286  (expect roughly 0.82-0.93 — LSTM often trails tree models slightly on this feature set)


## Step 8 — Save Models, Scalers & Test Data

This is the complete hand-off to Notebook 5 — nothing gets retrained there, only loaded and evaluated.

In [12]:
joblib.dump(xgb_model, 'models/forecast_xgb.pkl')
joblib.dump(lgbm_model, 'models/forecast_lgbm.pkl')
lstm_model.save('models/forecast_lstm.h5')
joblib.dump(scaler, 'models/forecast_scaler.pkl')
joblib.dump(y_scaler, 'models/forecast_yscaler.pkl')

joblib.dump({
    'X_test': X_test, 'X_test_sc': X_test_sc, 'X_test_seq': X_test_seq,
    'y_test': y_test, 'feature_cols': feature_cols, 'WINDOW': WINDOW,
    'X_train_sc': X_train_sc, 'y_train': y_train,
    'train_times': {'XGBoost': train_time_xgb, 'LightGBM': train_time_lgbm, 'LSTM': train_time_lstm}
}, 'models/forecast_test_data.pkl')

print("Saved 5 model/scaler files + test data bundle to models/")
for f in ['forecast_xgb.pkl','forecast_lgbm.pkl','forecast_lstm.h5','forecast_scaler.pkl',
          'forecast_yscaler.pkl','forecast_test_data.pkl']:
    assert os.path.exists(f'models/{f}'), f"Failed to save {f}"
print("Verified: all 6 files present. Ready for Notebook 5.")

Saved 5 model/scaler files + test data bundle to models/
Verified: all 6 files present. Ready for Notebook 5.
